# Supplementary Figure 4 — SWR controls

The submitted Extended Data Fig. 3 on non-overlapping windows, without its d (ISI entropy) and e (barrage-excised d2, now in Fig 3), with the revision's d2 and replay-probability panels added. d2 = d2 (2024 Euclidean, AR p = 20, 40 ms), labelled "d2". Panels carry stars only; session counts are in the figure legend. Every test is per session: the session is the unit.

- **a** pre vs post sleep d2, NREM, with SWR rate matched (per session, windows subsampled within 12 SWR-rate bins so both epochs have the same rate distribution): the submitted ED3a notebook (`matched_df`) on disjoint windows, tested by signed-rank across sessions.
- **b** pre vs post sleep, ripple-excised d2 (`_rippleablate.mat`), per-session means: the submitted ED3b notebook (`long`) on disjoint windows, drawn as Fig 2a and tested by signed-rank across sessions. Replaces the all-window Fig 2a companion panel.
- **c** d2 by SWR-rate sextile, full window (grey) and ripple-excised (red), 9 decoded sessions, non-overlapping NREM windows (`fig2_agh_excised.py` → `fig2_agh_nine_sessions.csv`, the table of Fig 2g); per-session Spearman, signed-rank across sessions for each arm.
- **d** replay probability by within-session d2 sextile, both arms (`fig2_new.panel_replay_prob`, unchanged; the excised arm recoloured from dark to Fig 2's red), tested per arm.
- **e, f** CA3 pyramidal FR by SWR rate, CA2 pyramidal FR by barrage rate (rat, 9 sessions each, FR z-scored per session): the submitted ED3f / ED3g panels on disjoint windows, drawing code unchanged. The bins are six equal-width bins of the session's rate (the notebook's `BINNING_METHOD = 'equal_width'`), not equal-count sextiles, so the axis says "bin" (f: 14 of 54 session × bin cells empty). Test changed: the submitted slope over the 6 pooled bin means (n = 6) is replaced by per-session Spearman ρ(bin, FR) and a signed-rank test across sessions. The drawn means are asserted equal to the per-session table the test uses. Error bars: s.e.m. with ddof = 1, as every other panel; the notebook's bars (ddof = 0) are asserted equal to that table's ddof = 0 value, then set to ddof = 1 (√(n/(n − 1)) longer). 9 sessions from 3 rats (AB4 ×4, AYA10 ×4, AYA7 ×1), the same sessions in e and f.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))   # notebooks/figures_revision
import numpy as np
import pandas as pd
import final_figs as F                     # paths, save/copy helpers; adds the revision folder to sys.path

## Data — the per-session tables (disjoint windows) and the session-level tests

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Supplementary Fig 4 = the submitted Extended Data Fig. 3 (SWR controls) on non-overlapping windows, without its
# d (ISI entropy) and e (barrage-excised d2, now Fig 3), with c and d of the revision added:
#   a  pre vs post d2, SWR rate matched        ED3a  Previous_Rv1 Fig2_A_FigS2_B_FigS3_A cell 5   matched_df
#   b  pre vs post d2, SWRs excised            ED3b  Previous_Rv1 FigS3_B_S3C_S3D_ripple_ablate cell 3   long
#   c  d2 by SWR-rate sextile, full + excised  fig2_agh_excised.py -> fig2_agh_nine_sessions.csv (as Fig 2g)
#   d  replay probability by d2 sextile        fig2_new.panel_replay_prob on excised_br_entropy_windows.csv
#   e  CA3 pyramidal FR by SWR-rate tile       ED3f  Previous_Rv1 FigS3_G_F_CA3_CA2_pyr_model_surpport_fig cell 3
#   f  CA2 pyramidal FR by barrage-rate tile   ED3g  same notebook, cell 6
# a, b, e, f: the submitted notebooks run as ed_nonoverlap.py runs them (every window table thinned to disjoint
# windows), each exporting the per-session table its panel plots (F.rv1_table) -- nothing re-implemented.
LOG = os.path.join(F.REPO, "data", "revision", "log")
T = {k: os.path.join(LOG, f"figs4{k}_{n}_disjoint.csv") for k, n in (
    ("a", "prepost_matched_ripple"), ("b", "prepost_ripple_excised"),
    ("e", "ca3_fr_by_swr_tile"), ("f", "ca2_fr_by_barrage_tile"))}
# e, f: the notebook's per-session tile means (pooled_data[col]: one dict per session, z-scored FR per tile),
# exported where each panel's cell starts, so that the test can be per session -- the submitted
# panel tested the slope of the 6 pooled tile means (n = 6)
TILES = ('pd.DataFrame([dict(n_points=s["n_points"], **{f"t{k + 1}": v for k, v in enumerate(s["tile_means_y"])}, '
         '**{f"n{k + 1}": c for k, c in enumerate(s["tile_counts"])}) for s in pooled_data[col_name]])')
JOBS = {"a": ("Fig2_A_FigS2_B_FigS3_A", "matched_df = pd.DataFrame(rows)", "matched_df"),
        "b": ("FigS3_B_S3C_S3D_ripple_ablate",
              'long["sleep_epoch_plot"] = long["sleep_epoch"].replace({"midsleep": "postsleep"})', "long"),
        "e": ("FigS3_G_F_CA3_CA2_pyr_model_surpport_fig", 'col_name = "CA3_E_FR_per_unit"', TILES),
        "f": ("FigS3_G_F_CA3_CA2_pyr_model_surpport_fig", 'col_name = "CA2_E_FR_per_unit"', TILES)}
for k, (nb, after, var) in JOBS.items():
    if not os.path.exists(T[k]):
        F.rv1_table(nb, after, var, T[k])

# a, b: one value per session and epoch (pre / post); the test is the session-paired signed-rank, as Fig 2a
A = pd.read_csv(T["a"]).set_index("animal_day")
PA = A.rename(columns={"d2_pre": "Presleep", "d2_post": "Postsleep"})[["Presleep", "Postsleep"]]
B = pd.read_csv(T["b"])
PB = (B.pivot(index="file_id", columns="sleep_epoch_plot", values="d2")
       .rename(columns={"presleep": "Presleep", "postsleep": "Postsleep"})[["Presleep", "Postsleep"]].dropna())


def paired(P):
    d = P.Postsleep - P.Presleep
    return dict(n=len(P), up=int((d > 0).sum()), mean_diff=float(d.mean()), sem=float(d.std(ddof=1) / np.sqrt(len(d))),
                p=float(stats.wilcoxon(P.Postsleep, P.Presleep).pvalue))


ST = {"a": paired(PA), "b": paired(PB)}


# e, f: per session, Spearman rho(tile, z-scored FR) over its non-empty tiles; signed-rank of the rhos across sessions
def tiles(path):
    D = pd.read_csv(path)
    Y = D[[f"t{k}" for k in range(1, 7)]].to_numpy(float)
    rho = np.array([stats.spearmanr(np.arange(1, 7)[np.isfinite(y)], y[np.isfinite(y)])[0] for y in Y])
    up = np.median(rho) > 0
    return D, Y, dict(n=len(rho), up=int((rho > 0).sum()) if up else int((rho < 0).sum()), direction="↑" if up else "↓",
                      rho_median=float(np.median(rho)), p=float(stats.wilcoxon(rho).pvalue),
                      empty_tiles=int((~np.isfinite(Y)).sum()))


DE, YE, ST["e"] = tiles(T["e"])
DF, YF, ST["f"] = tiles(T["f"])
for k, s in ST.items():
    print(f"{k}: two-sided Wilcoxon signed-rank p = {s['p']:.2g}; n = {s['n']} sessions")

## Panels a–f

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import same_color
from matplotlib.container import ErrorbarContainer
import fig2_new as N

H, W_BOX, W_LINE = 44.0, 44.0, 56.0                    # mm; three panels per row, as the submitted ED3


def legend_stats(ax, colors, loc):
    """each arm's statistic text ("9/9 ↑ **") moves into its legend entry, in the line's colour (as Fig 2c, 2g)"""
    st = list(ax.texts)
    hs, ls = ax.get_legend_handles_labels()
    assert len(st) == len(hs), (len(st), len(hs))
    for h, lab, t in zip(hs, ls, st):
        h.set_label(f"{lab}: {t.get_text().split(': ')[-1]}")
        t.remove()
    ax.legend(frameon=False, loc=loc)
    return colors


def finish(fig, ax, fid, colors=None):
    F.shrink(fig)
    if colors:
        for t, c in zip(ax.get_legend().get_texts(), colors):
            t.set_color(c)
    F.sparse_yticks(ax)
    F.settle(fig)
    F.drop_counts(fig)                                 # formal: stars only, counts in the legend
    F.show(F.save(fig, fid))


# ---- a, b: pre vs post sleep, per-session means -- the final Fig 2a panel (Fig2_A_prepost cell 5), on these tables
def prepost(P, s, fid, ylabel):
    F.final_style()
    S = P.rename_axis("session").reset_index().melt(id_vars="session", var_name="epoch2", value_name="d2")
    order, pal = ["Presleep", "Postsleep"], {"Presleep": "#f6d5ff", "Postsleep": "#dd55ff"}
    fig, ax = plt.subplots(figsize=(W_BOX * F.MM, H * F.MM), layout="constrained")
    sns.boxplot(data=S, x="epoch2", y="d2", order=order, hue="epoch2", hue_order=order, palette=pal, legend=False,
                showfliers=False, width=0.5, linewidth=0.6, ax=ax, zorder=2)
    sns.stripplot(data=S, x="epoch2", y="d2", order=order, color="k", jitter=False, alpha=0.75, size=2.2, ax=ax, zorder=3)
    for _, r in P.iterrows():
        ax.plot([0, 1], [r.Presleep, r.Postsleep], color="gray", alpha=0.5, lw=0.5, zorder=2)
    top = S.d2.max()
    ax.plot([0, 0, 1, 1], [top * 1.03, top * 1.05, top * 1.05, top * 1.03], color="k", lw=0.5)
    ax.text(0.5, top * 1.06, f"{s['up']}/{s['n']} ↑ {F.stars(s['p'])}", ha="center", va="bottom", fontsize=5.5)
    ax.set_ylim(top=top * 1.14)
    ax.set_xlabel(""); ax.set_ylabel(ylabel)
    F.settle(fig)
    F.drop_counts(fig)
    F.show(F.save(fig, fid))


prepost(PA, ST["a"], "FigS4_A_prepost_matched_swr_rate", "d2 (SWR rate matched)")
prepost(PB, ST["b"], "FigS4_B_prepost_ripple_excised", "d2 (ripple-excised)")

# ---- c: d2 by SWR-rate sextile, full window (grey) and ripple-excised (red): fig2_agh_excised.py's table, as Fig 2g
S = pd.read_csv(os.path.join(N.LOGDIR, "fig2_agh_nine_sessions.csv"))
F.native_style()
fig, ax = plt.subplots(figsize=(W_LINE * F.MM, H * F.MM), layout="constrained")
N.sextile_panel(ax, S, "d2_published", "d2", arms=(("full window", N.GREY),))
N.sextile_panel(ax, S, "d2_excised", "d2", arms=(("ripple-excised", N.RED),))
ax.set_xlabel("SWR-rate sextile")
finish(fig, ax, "FigS4_C_d2_by_swr_rate", legend_stats(ax, [N.GREY, N.RED], "upper left"))
L_C = [l_ for l_ in ax.get_lines() if l_.get_marker() == "o"][0]          # c's drawn data line: e and f match it
LW, MS = L_C.get_linewidth(), L_C.get_markersize()
CAP_MS = max(l_.get_markersize() for l_ in ax.get_lines() if l_.get_marker() != "o")

# ---- d: replay probability by within-session d2 sextile, both arms: fig2_new.panel_replay_prob, unchanged; its
# excised arm (drawn in DARK) recoloured to Fig 2's red so that c and d read alike
W = pd.read_csv(os.path.join(N.LOGDIR, "excised_br_entropy_windows.csv"))
F.native_style()
fig, ax = plt.subplots(figsize=(W_LINE * F.MM, H * F.MM), layout="constrained")
N.panel_replay_prob(ax, W)
for a_ in ax.get_lines():
    if same_color(a_.get_color(), N.DARK):
        a_.set_color(N.RED); a_.set_markerfacecolor(N.RED); a_.set_markeredgecolor(N.RED)
for c_ in ax.collections:
    if len(c_.get_colors()) and same_color(c_.get_colors()[0], N.DARK):
        c_.set_color(N.RED)
for t in ax.texts:
    t.set_text(t.get_text().replace("full-window d2", "full window").replace("ripple-excised d2", "ripple-excised"))
# stars: the submitted Fig 2d linear mixed model on these non-overlapping windows, random intercept and slope per
# session (ed4d_replay_prob_lmm.py); the counts stay the per-session Spearman direction
import re as _re
LMM = pd.read_csv(os.path.join(N.LOGDIR, "ed4d_replay_prob_lmm.csv"))
LMM = LMM[(LMM.data == "non-overlapping (ED 4d)") & (LMM.model == "random intercept + slope")].set_index("arm").p
for t in ax.texts:
    arm = "full window" if t.get_text().startswith("full") else "ripple-excised"
    t.set_text(_re.sub(r"(n\.s\.|\*+)\s*$", F.stars(LMM[arm]), t.get_text()))
    print(f"d {arm}: linear mixed model (random intercept and slope per session) p = {LMM[arm]:.2g}")
for h in ax.get_legend_handles_labels()[0]:
    h.set_label(h.get_label().replace("full-window d2", "full window").replace("ripple-excised d2", "ripple-excised"))
ax.set_ylabel("Replay probability")          # short y; "pp relative to the session mean" is in the legend
ax.set_xlabel("d2 sextile (within session)")
finish(fig, ax, "FigS4_D_replay_prob_by_d2", legend_stats(ax, [N.GREY, N.RED], "upper left"))

# ---- e, f: the submitted ED3f / ED3g panels (their drawing code unchanged, captured and restyled to print size), with
# the per-session test: Spearman rho(bin, FR) per session, signed-rank across sessions (data step). The bins are SIX
# EQUAL-WIDTH bins of the session's rate (the notebook's BINNING_METHOD = 'equal_width'), not equal-count sextiles.
CAP = F.run_rv1_capture("FigS3_G_F_CA3_CA2_pyr_model_surpport_fig",
                        ["S3_F_pooled_CA3_E_FR_per_unit_vs_ripple_tiles", "S3_G_model_pooled_CA2_E_FR_per_unit_vs_barrage_tiles"])
for key, Y, s, fid, xl, yl in (
        ("S3_F_pooled_CA3_E_FR_per_unit_vs_ripple_tiles", YE, ST["e"], "FigS4_E_ca3_fr_by_swr_rate", "SWR-rate bin",
         "CA3 pyramidal FR (z)"),
        ("S3_G_model_pooled_CA2_E_FR_per_unit_vs_barrage_tiles", YF, ST["f"], "FigS4_F_ca2_fr_by_barrage_rate",
         "BARR-rate bin", "CA2 pyramidal FR (z)")):
    fig = CAP[key]
    ax = fig.axes[0]
    line = [l_ for l_ in ax.get_lines() if len(l_.get_xdata()) >= 4][0]
    assert np.allclose(line.get_ydata(), np.nanmean(Y, 0)[np.isfinite(np.nanmean(Y, 0))]), key   # same table as drawn
    if ax.get_legend():
        ax.get_legend().remove()                       # the notebook's empty legend box
    # s.e.m. with ddof = 1, as every other panel; the notebook's bars use np.nanstd (ddof = 0).
    # The drawn bars are first asserted to be the ddof = 0 value of the per-session table, then set to ddof = 1.
    m, nv = np.nanmean(Y, 0), np.isfinite(Y).sum(0)
    ok = np.isfinite(m)
    se0, se1 = np.nanstd(Y, 0) / np.sqrt(nv), np.nanstd(Y, 0, ddof=1) / np.sqrt(nv)
    ec = [c_ for c_ in ax.containers if isinstance(c_, ErrorbarContainer)][0]
    _, caps, bars = ec.lines
    seg = bars[0].get_segments()
    assert np.allclose([s_[1][1] - s_[0][1] for s_ in seg], 2 * se0[ok]), key
    xs = np.arange(1, 7)[ok]
    bars[0].set_segments([[[x, y - e], [x, y + e]] for x, y, e in zip(xs, m[ok], se1[ok])])
    lo, hi = sorted(caps, key=lambda c_: np.mean(c_.get_ydata()))
    lo.set_ydata(m[ok] - se1[ok]); hi.set_ydata(m[ok] + se1[ok])
    y0, y1 = ax.get_ylim()
    ax.set_ylim(min(y0, np.min(m[ok] - se1[ok]) - 0.05 * (y1 - y0)), max(y1, np.max(m[ok] + se1[ok]) + 0.05 * (y1 - y0)))
    F.restyle(fig, W_LINE, H)
    ax.set_title("")                                   # the notebook's title: its pooled Pearson r over the 6 bins (n = 6)
    for l_ in ax.get_lines():                          # line weight, markers and caps as c's drawn line
        l_.set_linewidth(LW)
        l_.set_markersize(MS if l_.get_marker() == "o" else CAP_MS)
        l_.set_markeredgewidth(LW)
    for c_ in ax.collections:
        c_.set_linewidth(LW)
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.text(0.05, 0.95, f"{s['up']}/{s['n']} {s['direction']} {F.stars(s['p'])}", transform=ax.transAxes,
            ha="left", va="top", fontsize=6, color=line.get_color())
    F.sparse_yticks(ax)
    F.settle(fig)
    F.drop_counts(fig)
    F.show(F.save(fig, fid))